In [1]:
# CELL 1 — Install Dependencies
import subprocess
import sys

packages = [
    "fastapi",
    "uvicorn[standard]",
    "pydantic",
    "nest_asyncio",
    "pyngrok"
]

print("📦 Memeriksa dependencies...")
for pkg in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    except Exception as e:
        print(f"Peringatan install {pkg}: {e}")
print("✅ Semua dependensi siap!")


📦 Memeriksa dependencies...
✅ Semua dependensi siap!


In [2]:
# CELL 2 — Imports & Konfigurasi Global
import os
import re
import html
import unicodedata
import warnings
from enum import Enum
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")

# ── Enum Parameter Klasifikasi & Aksi Sistem ─────────────────────────
class CommentCategory(str, Enum):
    POSITIF = "Kritik & Saran Positif"
    ANEH_KASAR = "Comment aneh-aneh (komentar yang berbahasa kasar)"
    NEGATIF = "Kritik & Saran Negatif"


class ModerationAction(str, Enum):
    DELIVERED_TO_DASHBOARD = "DELIVERED_TO_DASHBOARD"
    AUTO_REMOVED = "AUTO_REMOVED"


class SentimentType(str, Enum):
    POSITIVE = "POSITIVE"
    NEGATIVE = "NEGATIVE"
    TOXIC = "TOXIC"


print("✅ Enum parameter klasifikasi berhasil dikonfigurasi:")
for cat in CommentCategory:
    print(f" • {cat.value}")


✅ Enum parameter klasifikasi berhasil dikonfigurasi:
 • Kritik & Saran Positif
 • Comment aneh-aneh (komentar yang berbahasa kasar)
 • Kritik & Saran Negatif


In [3]:
# CELL 3 — Kamus & Lexicon Knowledge Base Bahasa Indonesia
# (Slang, Makian Kasar, Pola Sensor Bintang, Aspek, dan Leksikon Sentimen)

# 1. Kamus Slang & Singkatan Gaul Bahasa Indonesia
SLANG_MAP = {
    "bgt": "banget", "bgtss": "banget", "bgtt": "banget", "bngtt": "banget",
    "ga": "tidak", "gak": "tidak", "gk": "tidak", "g": "tidak", "kaga": "tidak",
    "kagak": "tidak", "tdk": "tidak", "ngga": "tidak", "nggak": "tidak", "nda": "tidak",
    "tp": "tapi", "tpi": "tapi", "ttp": "tetap",
    "dgn": "dengan", "dg": "dengan",
    "sy": "saya", "aq": "saya", "aku": "saya", "gw": "saya", "gue": "saya", "gua": "saya",
    "lu": "kamu", "lo": "kamu", "loe": "kamu", "ente": "kamu", "lw": "kamu",
    "bgs": "bagus", "bgus": "bagus", "mantul": "mantap", "mantep": "mantap",
    "mantapp": "mantap", "top": "mantap", "jos": "mantap", "josss": "mantap",
    "krg": "kurang", "kurg": "kurang", "kurng": "kurang",
    "bnyk": "banyak", "byk": "banyak",
    "dikit": "sedikit", "sdkt": "sedikit", "dkit": "sedikit",
    "tlg": "tolong", "pls": "tolong", "plz": "tolong", "mohon": "tolong",
    "mkn": "makan", "mknn": "makanan", "makann": "makanan",
    "minum": "minuman", "mnm": "minuman", "mnmn": "minuman",
    "pesen": "pesan", "pesenan": "pesanan", "orderan": "pesanan",
    "hrg": "harga", "mrh": "murah", "mhl": "mahal",
    "packing": "kemasan", "packaging": "kemasan", "bungkus": "kemasan", "bungkusnya": "kemasan",
    "ancur": "hancur", "tumpah": "tumpah", "bocor": "bocor", "robek": "robek",
    "asin": "asin", "keasinan": "asin", "hambar": "hambar", "anyep": "hambar",
    "basi": "basi", "bau": "bau", "apek": "bau", "mentah": "mentah",
    "gosong": "gosong", "hangus": "gosong",
    "ramah": "ramah", "judes": "judes", "jutek": "jutek", "galak": "galak", "cuek": "cuek",
    "cepet": "cepat", "cpt": "cepat", "fast": "cepat", "kenceng": "cepat",
    "lama": "lambat", "lemot": "lambat", "lelet": "lambat", "llet": "lambat",
    "recomended": "rekomendasi", "rekomen": "rekomendasi", "worth": "layak", "worthit": "layak",
    "nyesel": "menyesal", "kecewa": "kecewa", "kapok": "kapok", "zonk": "zonk",
    "bikin": "membuat", "beneran": "sungguh",
    "next": "selanjutnya", "kedepan": "kedepannya", "kalo": "kalau", "kl": "kalau", "klo": "kalau"
}

# 2. Kamus Kata Kasar & Makian Bahasa Indonesia (Toxic Lexicon)
ABUSIVE_WORDS = {
    # Hewan sebagai makian
    "anjing", "anj1ng", "anjink", "anying", "anjeng", "asw", "asu",
    "babi", "b4b1", "bb1",
    # Makian kasar & umpatan
    "bangsat", "b@ngs@t", "b4ngs4t", "bajingan", "b4j1ng4n", "kampret", "brengsek", "keparat",
    "jancuk", "jancok", "dancok", "dancuk", "cok", "cuk",
    "bangke", "bangkai", "tai", "taek", "tae", "setan", "iblis", "dajjal",
    # Organ reproduksi & seksual kasar
    "kontol", "k0nt0l", "kntil", "kntl", "memek", "mmk", "jembut", "jmbt",
    "pantek", "peler", "itil", "tetek", "ngentot", "ngentod",
    "lonte", "lont3", "pelacur", "jablay", "perek", "sundal", "pecun", "bencong",
    # Cacian mental/intelektual kasar
    "goblok", "g0bl0k", "gblk", "goblog", "tolol", "t0l0l", "tll", "bego", "bgok",
    "idiot", "pekok", "autis", "dungu", "otak udang",
    # Frasa umpatan
    "sialan", "biadab", "laknat", "mampus", "modar", "mati aja"
}

# 3. Pola Regex untuk Makian Bertanda Bintang / Sensor Simbol (misal: baji***an, anj**g)
MASKED_ABUSIVE_PATTERNS = [
    r'\bb+a+j+i+[\*\@\#\$\%\&]+[a-z]*\b',   # baji***an
    r'\ba+n+j+[\*\@\#\$\%\&]+[a-z]*\b',     # anj**g, anj**k
    r'\bb+[\*\@\#\$\%\&]+b+i+\b',           # b*bi, b**i
    r'\bk+[\*\@\#\$\%\&]+n+t+[\*\@\#\$\%\&]*o*l*\b', # k*nt*l
    r'\bg+[\*\@\#\$\%\&]+b+l+[\*\@\#\$\%\&]*[a-z]*\b', # g*bl*k
    r'\bb+[\*\@\#\$\%\&]+n+g+s+[\*\@\#\$\%\&]*t*\b',   # b*ngs*t
    r'\bm+[\*\@\#\$\%\&]+m+[\*\@\#\$\%\&]*k+\b',       # m*m*k
    r'\bt+[\*\@\#\$\%\&]+l+[\*\@\#\$\%\&]*l*\b',       # t*l*l
    r'\bj+a+n+c+[\*\@\#\$\%\&]+[a-z]*\b',   # janc**k
]

# 4. Kamus Aspek Review untuk Analisis Dashboard Toko
ASPECT_KEYWORDS = {
    "Rasa & Kualitas": [
        "rasa", "rasanya", "enak", "mantap", "lezat", "sedap", "nagih", "nikmat",
        "asin", "keasinan", "hambar", "manis", "kemanisan", "pahit", "pedas", "pedes",
        "basi", "bau", "mentah", "gosong", "dingin", "renyah", "crispy", "alot", "segar"
    ],
    "Porsi & Takaran": [
        "porsi", "porsinya", "dikit", "sedikit", "banyak", "kenyang", "pas",
        "jumbo", "mini", "ukurannya", "potongan", "isi", "isinya", "tebel", "tipis"
    ],
    "Kemasan & Kebersihan": [
        "kemasan", "packing", "packaging", "bungkus", "bungkusnya", "plastik",
        "tumpah", "bocor", "robek", "rapi", "higienis", "bersih", "kotor", "debu", "paper bag"
    ],
    "Pelayanan & Tenant": [
        "pelayanan", "penjual", "seller", "admin", "ramah", "sopan", "jutek",
        "galak", "judes", "respon", "chat", "dibalas", "dibales", "cuek", "tanggap"
    ],
    "Kecepatan & Pengiriman": [
        "cepat", "cepet", "kilat", "lama", "lambat", "lelet", "nunggu", "antri",
        "antar", "pengiriman", "sampai", "jam"
    ],
    "Harga & Nilai": [
        "harga", "harganya", "murah", "terjangkau", "mahal", "kemahalan",
        "worth", "worthit", "sebanding", "sesuai", "promo", "diskon"
    ]
}

# 5. Indikator Saran / Rekomendasi Konstruktif
SUGGESTION_KEYWORDS = [
    "saran", "sarannya", "masukan", "mungkin", "bisa", "tolong", "mohon",
    "ke depannya", "kedepannya", "next", "perlu", "ditingkatkan", "ditambah",
    "variasi", "varian", "opsi", "coba", "bagus kalau", "lebih baik", "harap"
]

# 6. Sentiment Lexicon
POSITIVE_WORDS = {
    "enak", "mantap", "lezat", "nikmat", "sedap", "suka", "favorit", "nagih",
    "juara", "top", "bagus", "rapi", "bersih", "ramah", "cepat", "puas",
    "murah", "terjangkau", "rekomendasi", "rekomen", "terbaik", "senang",
    "berlangganan", "langganan", "memuaskan", "segar", "hangat", "krispi",
    "pas", "worth", "worthit", "oke", "ok", "jempol", "keren", "estetik",
    "higienis", "fresh", "crispy"
}

NEGATIVE_WORDS = {
    "kurang", "kecewa", "menyesal", "nyesel", "kapok", "zonk",
    "asin", "keasinan", "hambar", "basi", "bau", "mentah", "gosong",
    "tumpah", "bocor", "kotor", "dingin", "alot", "keras", "lama", "lelet",
    "lambat", "mahal", "kemahalan", "dikit", "sedikit", "rusak", "tertukar",
    "judes", "jutek", "galak", "cuek", "penipu", "nipu", "bohong",
    "tidak ramah", "ga ramah", "tidak enak", "ga enak", "nggak enak",
    "tidak dibalas", "ga dibales", "diabaikan", "tidak niat", "ga niat"
}

print(f"✅ Kamus NLU selesai dimuat:")
print(f" • Total kata kasar dimonitor: {len(ABUSIVE_WORDS)} kata")
print(f" • Total kata slang dinormalisasi: {len(SLANG_MAP)} kata")
print(f" • Total kategori aspek ulasan: {len(ASPECT_KEYWORDS)} aspek")


✅ Kamus NLU selesai dimuat:
 • Total kata kasar dimonitor: 74 kata
 • Total kata slang dinormalisasi: 108 kata
 • Total kategori aspek ulasan: 6 aspek


In [ ]:
# CELL 4 — Preprocessing & Leetspeak Normalizer Engine

def normalize_leetspeak(text: str) -> str:
    """
    Mengembalikan teks leetspeak / angka pengganti huruf ke karakter alfabet standar.
    Contoh: 'anj1ng' -> 'anjing', 'g0bl0k' -> 'goblok', 'k0nt0l' -> 'kontol', 'b.a.b.i' -> 'babi'.
    """
    # Hapus spasi atau titik pemisah sengaja antar huruf (misal: 'a.n.j.i.n.g' -> 'anjing')
    de_spaced = re.sub(r'(?<=\b[a-zA-Z0-9])[\.\-\_\s](?=[a-zA-Z0-9]\b)', '', text)

    leet_map = {
        '0': 'o', '1': 'i', '3': 'e', '4': 'a', '5': 's',
        '7': 't', '8': 'b', '@': 'a', '$': 's', '!': 'i'
    }
    res = []
    for char in de_spaced:
        res.append(leet_map.get(char, char))
    return "".join(res)


def preprocess_text(text: str) -> Tuple[str, str]:
    """
    Pipeline NLU Preprocessing:
    1. HTML Unescape & Lowercasing
    2. Leetspeak Decoding
    3. Collapsing Karakter Berulang (misal: 'enaaaak' 'enak')
    4. Slang/Abbreviation Expansion ke bahasa Indonesia formal/baku
    Mengembalikan: (raw_cleaned_text, normalized_text_for_analysis)
    """
    if not text:
        return "", ""

    # 1. Clean & Lowercase
    raw = html.unescape(text).strip()
    lowered = unicodedata.normalize('NFKD', raw).lower()

    # 2. Leetspeak Decoding
    normalized_leet = normalize_leetspeak(lowered)

    # 3. Collapse repeated chars (3x berturut-turut jadi 1)
    collapsed = re.sub(r'(.)\1{2,}', r'\1', normalized_leet)

    # 4. Tokenize & Slang Expansion
    tokens = re.findall(r"\b[\w'-]+\b", collapsed)
    expanded_tokens = [SLANG_MAP.get(tok, tok) for tok in tokens]
    expanded_text = " ".join(expanded_tokens)

    return raw, expanded_text


# Tes Cepat Preprocessing
sample_raw = "Woy penjual anj1ng, makanannya ga enak bgtss & kuah tumpaaah!!"
_, sample_clean = preprocess_text(sample_raw)
print("Original:", sample_raw)
print("Cleaned :", sample_clean)


Original: Woy penjual anj1ng, makanannya ga enak bgtss & kuah tumpaaah!!
Cleaned : woy penjual anjing makanannya tidak enak banget kuah tumpahii


In [5]:
# CELL 5 — Stage 1: Safety & Toxicity Gate (Filter Kasar / Aneh-aneh)

# Compile regex pattern untuk pencarian cepat & boundary-aware
escaped_abusive = [re.escape(w) for w in ABUSIVE_WORDS]
abusive_regex = re.compile(r'\b(' + '|'.join(escaped_abusive) + r')\b', re.IGNORECASE)
masked_regexes = [re.compile(p, re.IGNORECASE) for p in MASKED_ABUSIVE_PATTERNS]


def detect_profanity_and_toxicity(raw_text: str, preprocessed_text: str) -> Tuple[bool, List[str], str]:
    """
    Mengecek apakah komentar mengandung kata-kata kasar, makian, ujaran kebencian,
    sensoran leetspeak, atau konten toksik ('Comment aneh-aneh').
    """
    flagged = []
    lowered_raw = raw_text.lower()
    leet_normalized = normalize_leetspeak(lowered_raw)

    # 1. Cek kata kasar standar pada teks raw, leet, dan preprocessed
    for target in [lowered_raw, leet_normalized, preprocessed_text]:
        matches = abusive_regex.findall(target)
        for m in matches:
            clean_m = m.lower()
            if clean_m not in flagged:
                flagged.append(clean_m)

    # 2. Cek makian bertanda bintang / simbol (misal: baji***an, a**ing, k*nt*l)
    for pattern in masked_regexes:
        matches = pattern.findall(lowered_raw)
        for m in matches:
            clean_m = m.lower()
            if clean_m not in flagged:
                flagged.append(f"sensoran ({clean_m})")

    # 3. Cek context-dependent insults (contoh: 'toko sampah', 'makanan sampah', 'najis')
    if re.search(r'\b(sampah|najis)\b', preprocessed_text):
        flagged.append("hinaan degradatif (sampah/najis)")

    if flagged:
        reason = f"Terdeteksi kata-kata kasar / makian: {', '.join(flagged)}. Sesuai aturan moderasi, komentar otomatis di-remove."
        return True, flagged, reason

    return False, [], ""


# Uji Coba Safety Gate
for txt in ["Penjual ramah sekali", "Woy babi lu jualan basi anj1ng!", "makanan baji***an nipu"]:
    _, clean = preprocess_text(txt)
    toxic, flags, reason = detect_profanity_and_toxicity(txt, clean)
    print(f"Teks: '{txt}'")
    print(f" ↳ Kasar/Toxic: {toxic} | Flagged: {flags}")
    print("-" * 50)


Teks: 'Penjual ramah sekali'
 ↳ Kasar/Toxic: False | Flagged: []
--------------------------------------------------
Teks: 'Woy babi lu jualan basi anj1ng!'
 ↳ Kasar/Toxic: True | Flagged: ['babi', 'anj1ng']
--------------------------------------------------
Teks: 'makanan baji***an nipu'
 ↳ Kasar/Toxic: True | Flagged: ['sensoran (baji***an)']
--------------------------------------------------


In [6]:
# CELL 6 — Stage 2 & 3: Sentiment, Suggestion & Aspect Classifier Engine

def extract_aspects(text: str) -> List[str]:
    """Mengekstraksi aspek-aspek apa saja yang dibahas dalam komentar."""
    found_aspects = []
    text_lower = text.lower()
    for aspect, keywords in ASPECT_KEYWORDS.items():
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
                found_aspects.append(aspect)
                break
    return found_aspects


def classify_comment(
    text: str,
    comment_id: Optional[str] = None,
    tenant_id: Optional[str] = None,
    rating: Optional[float] = None
) -> Dict[str, Any]:
    """
    Fungsi Utama Klasifikasi & Moderasi NLU:
    Mengembalikan keputusan routing:
    - Jika Kategori 'Comment aneh-aneh' -> Action: AUTO_REMOVED (Dihapus dari publik, tidak ke dashboard)
    - Jika Kategori 'Kritik & Saran Positif' -> Action: DELIVERED_TO_DASHBOARD (Terkirim ke dashboard)
    - Jika Kategori 'Kritik & Saran Negatif' -> Action: DELIVERED_TO_DASHBOARD (Terkirim ke dashboard)
    """
    raw_text, preprocessed_text = preprocess_text(text)

    # Validasi input kosong
    if not raw_text.strip():
        return {
            "comment_id": comment_id,
            "tenant_id": tenant_id,
            "original_text": text,
            "cleaned_text": "",
            "category": CommentCategory.NEGATIF.value,
            "action": ModerationAction.AUTO_REMOVED.value,
            "is_delivered_to_dashboard": False,
            "confidence_score": 1.0,
            "sentiment": SentimentType.NEGATIVE.value,
            "aspects": [],
            "flagged_words": [],
            "moderation_reason": "Komentar kosong / tidak valid.",
            "dashboard_badge": "DIBLOKIR / REMOVED"
        }

    # ── STAGE 1: Toxicity & Safety Gate ─────────────────────────
    is_toxic, flagged_words, toxic_reason = detect_profanity_and_toxicity(raw_text, preprocessed_text)

    if is_toxic:
        return {
            "comment_id": comment_id,
            "tenant_id": tenant_id,
            "original_text": raw_text,
            "cleaned_text": preprocessed_text,
            "category": CommentCategory.ANEH_KASAR.value,
            "action": ModerationAction.AUTO_REMOVED.value,
            "is_delivered_to_dashboard": False,
            "confidence_score": 0.98,
            "sentiment": SentimentType.TOXIC.value,
            "aspects": extract_aspects(preprocessed_text),
            "flagged_words": flagged_words,
            "moderation_reason": toxic_reason,
            "dashboard_badge": "DIBLOKIR / REMOVED"
        }

    # ── STAGE 2: Sentiment & Constructive Suggestion Classification ──
    pos_score = 0.0
    neg_score = 0.0
    suggestion_score = 0.0

    tokens = preprocessed_text.split()
    for tok in tokens:
        if tok in POSITIVE_WORDS:
            pos_score += 1.0
        if tok in NEGATIVE_WORDS:
            neg_score += 1.0
        if tok in SUGGESTION_KEYWORDS:
            suggestion_score += 0.8

    # Cek frasa negasi negatif (misal: 'tidak enak', 'kurang manis', 'tidak dibalas')
    if re.search(r'\b(tidak|kurang|bukan|belum|ga|gak|nggak)\s+(enak|mantap|manis|sedap|rapi|cepat|bersih|puas|ramah|dibalas|dibales)\b', preprocessed_text):
        neg_score += 2.5

    # Cek frasa keluhan tajam (misal: 'nyesel beli', 'kecewa banget', 'nipu lu')
    if re.search(r'\b(nyesel|kecewa|kapok|nipu|judes|jutek|zonk|hambar|basi|tumpah)\b', preprocessed_text):
        neg_score += 2.0

    # Cek frasa penguat positif (misal: 'enak banget', 'sangat ramah')
    if re.search(r'\b(enak|mantap|lezat|puas|ramah|cepat|segar|estetik)\s+(banget|sekali|pisan)\b', preprocessed_text):
        pos_score += 2.5

    # Pertimbangan Rating (jika ada nilai rating 1-5 bintang dari pembeli)
    if rating is not None:
        if rating >= 4.0:
            pos_score += 1.5
        elif rating <= 2.5:
            neg_score += 1.5

    # Aspek yang dibahas
    aspects = extract_aspects(preprocessed_text)

    # Saran positif membangun tanpa keluhan negatif tajam -> masuk ke Kritik & Saran Positif
    if suggestion_score > 0 and neg_score == 0:
        pos_score += 1.0

    if pos_score >= neg_score:
        # Kritik & Saran Positif
        total = max(pos_score + neg_score + 0.1, 1.0)
        confidence = min(round(0.72 + (pos_score / total) * 0.26, 2), 0.99)
        category = CommentCategory.POSITIF.value
        sentiment = SentimentType.POSITIVE.value
        action = ModerationAction.DELIVERED_TO_DASHBOARD.value
        is_delivered = True
        reason = "Komentar berupa feedback apresiasi atau saran positif yang membangun. Diteruskan ke dashboard toko."
        badge = "Apresiasi & Rekomendasi"
    else:
        # Kritik & Saran Negatif
        total = max(pos_score + neg_score + 0.1, 1.0)
        confidence = min(round(0.72 + (neg_score / total) * 0.26, 2), 0.99)
        category = CommentCategory.NEGATIF.value
        sentiment = SentimentType.NEGATIVE.value
        action = ModerationAction.DELIVERED_TO_DASHBOARD.value
        is_delivered = True
        reason = "Komentar berisi kritik/keluhan santun tanpa bahasa kasar. Diteruskan ke dashboard toko untuk evaluasi perbaikan."
        badge = "Perlu Perhatian / Evaluasi"

    return {
        "comment_id": comment_id,
        "tenant_id": tenant_id,
        "original_text": raw_text,
        "cleaned_text": preprocessed_text,
        "category": category,
        "action": action,
        "is_delivered_to_dashboard": is_delivered,
        "confidence_score": confidence,
        "sentiment": sentiment,
        "aspects": aspects,
        "flagged_words": [],
        "moderation_reason": reason,
        "dashboard_badge": badge
    }


def moderate_batch(comments: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Moderasi dan klasifikasi kumpulan komentar sekaligus."""
    processed_comments = []
    delivered_to_dashboard = []
    auto_removed = []

    summary_counts = {
        CommentCategory.POSITIF.value: 0,
        CommentCategory.NEGATIF.value: 0,
        CommentCategory.ANEH_KASAR.value: 0,
    }

    for item in comments:
        cid = item.get("id") or item.get("comment_id")
        tid = item.get("tenant_id") or item.get("tenantId")
        text = item.get("text") or item.get("comment") or item.get("comment_text") or ""
        rating = item.get("rating")

        result = classify_comment(text=text, comment_id=cid, tenant_id=tid, rating=rating)
        processed_comments.append(result)
        summary_counts[result["category"]] += 1

        if result["is_delivered_to_dashboard"]:
            delivered_to_dashboard.append(result)
        else:
            auto_removed.append(result)

    return {
        "total_processed": len(comments),
        "total_delivered_to_dashboard": len(delivered_to_dashboard),
        "total_auto_removed": len(auto_removed),
        "summary_by_category": summary_counts,
        "delivered_comments": delivered_to_dashboard,
        "removed_comments": auto_removed,
        "all_results": processed_comments
    }

print("✅ Core NLU Classifier & Batch Moderator siap digunakan!")


✅ Core NLU Classifier & Batch Moderator siap digunakan!


In [7]:
# CELL 7 — Pydantic Schemas (Request & Response Model API)
from pydantic import BaseModel, Field

class CommentClassifyRequest(BaseModel):
    comment_text: str = Field(..., description="Teks ulasan pembeli untuk toko", example="Makanannya enak banget, porsinya pas!")
    comment_id: Optional[str] = Field(None, description="ID ulasan dari database", example="cmt-101")
    tenant_id: Optional[str] = Field(None, description="ID tenant/toko penjual", example="tenant-7")
    rating: Optional[float] = Field(None, ge=1.0, le=5.0, description="Rating bintang ulasan 1-5")


class CommentClassifyResponse(BaseModel):
    comment_id: Optional[str] = None
    tenant_id: Optional[str] = None
    original_text: str
    cleaned_text: str
    category: str = Field(..., description="Kategori klasifikasi ulasan")
    action: str = Field(..., description="Aksi: DELIVERED_TO_DASHBOARD atau AUTO_REMOVED")
    is_delivered_to_dashboard: bool = Field(..., description="Status apakah ulasan diteruskan ke dashboard penjual")
    confidence_score: float = Field(..., description="Skor kepercayaan model [0-1]")
    sentiment: str = Field(..., description="Sentimen ulasan: POSITIVE, NEGATIVE, TOXIC")
    aspects: List[str] = Field(default_factory=list, description="Aspek-aspek yang dibahas")
    flagged_words: List[str] = Field(default_factory=list, description="Kata-kata kasar yang terdeteksi")
    moderation_reason: str = Field(..., description="Alasan moderasi sistem")
    dashboard_badge: Optional[str] = Field(None, description="Badge UI untuk dashboard penjual")


class CommentBatchItem(BaseModel):
    comment_id: Optional[str] = None
    tenant_id: Optional[str] = None
    comment_text: str
    rating: Optional[float] = None


class CommentBatchModerateRequest(BaseModel):
    comments: List[CommentBatchItem]


class CommentBatchModerateResponse(BaseModel):
    total_processed: int
    total_delivered_to_dashboard: int
    total_auto_removed: int
    summary_by_category: Dict[str, int]
    delivered_comments: List[CommentClassifyResponse]
    removed_comments: List[CommentClassifyResponse]
    all_results: List[CommentClassifyResponse]

print("✅ Pydantic Schemas siap untuk validasi API!")


✅ Pydantic Schemas siap untuk validasi API!


In [8]:
# CELL 8 — FastAPI App + Jalankan Server di Notebook
import nest_asyncio
import uvicorn
import threading
import time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import RedirectResponse

nest_asyncio.apply()

app = FastAPI(
    title="Tenantify AI — NLU Comment Moderation API",
    description="Microservice klasifikasi ulasan toko berbasis NLU untuk memoderasi kata kasar dan meneruskan kritik positif/negatif ke dashboard.",
    version="1.0.0",
    docs_url="/docs"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/", include_in_schema=False)
def root():
    return RedirectResponse(url="/docs")

@app.post("/comments/classify", response_model=CommentClassifyResponse, tags=["Comment NLU Moderation"])
def api_classify_comment(req: CommentClassifyRequest):
    """Klasifikasi ulasan toko secara real-time."""
    result = classify_comment(
        text=req.comment_text,
        comment_id=req.comment_id,
        tenant_id=req.tenant_id,
        rating=req.rating
    )
    return CommentClassifyResponse(**result)

@app.post("/comments/moderate-batch", response_model=CommentBatchModerateResponse, tags=["Comment NLU Moderation"])
def api_moderate_batch(req: CommentBatchModerateRequest):
    """Batch moderasi banyak ulasan toko sekaligus."""
    raw_list = [
        {
            "id": c.comment_id,
            "tenant_id": c.tenant_id,
            "text": c.comment_text,
            "rating": c.rating
        }
        for c in req.comments
    ]
    batch_res = moderate_batch(raw_list)
    return CommentBatchModerateResponse(
        total_processed=batch_res["total_processed"],
        total_delivered_to_dashboard=batch_res["total_delivered_to_dashboard"],
        total_auto_removed=batch_res["total_auto_removed"],
        summary_by_category=batch_res["summary_by_category"],
        delivered_comments=[CommentClassifyResponse(**c) for c in batch_res["delivered_comments"]],
        removed_comments=[CommentClassifyResponse(**c) for c in batch_res["removed_comments"]],
        all_results=[CommentClassifyResponse(**c) for c in batch_res["all_results"]],
    )

@app.get("/comments/categories", tags=["Comment NLU Moderation"])
def api_categories():
    """Metadata 3 parameter klasifikasi ulasan."""
    return {
        "categories": [
            {
                "name": CommentCategory.POSITIF.value,
                "action": ModerationAction.DELIVERED_TO_DASHBOARD.value,
                "target": "Dashboard Toko / Penjual",
                "description": "Komentar apresiasi, pujian rasa/kemasan/pelayanan, atau saran positif yang membangun.",
                "badge": "Apresiasi & Rekomendasi"
            },
            {
                "name": CommentCategory.NEGATIF.value,
                "action": ModerationAction.DELIVERED_TO_DASHBOARD.value,
                "target": "Dashboard Toko / Penjual",
                "description": "Kritik atau keluhan santun mengenai kualitas makanan/pelayanan/pengiriman tanpa kata kasar untuk evaluasi toko.",
                "badge": "Perlu Perhatian / Evaluasi"
            },
            {
                "name": CommentCategory.ANEH_KASAR.value,
                "action": ModerationAction.AUTO_REMOVED.value,
                "target": "Otomatis Dihapus (Auto-Removed)",
                "description": "Komentar toxic yang mengandung makian, kata kasar, pelecehan, atau sensor leetspeak kasar.",
                "badge": "DIBLOKIR / REMOVED"
            }
        ]
    }

def run_server(port=8001):
    config = uvicorn.Config(app=app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    server.run()

# Jalankan server FastAPI di background thread pada port 8001
server_port = 8001
t = threading.Thread(target=run_server, args=(server_port,), daemon=True)
t.start()
time.sleep(1.5)

print(f"🚀 Server NLU Moderation berhasil berjalan di background!")
print(f"📖 Swagger Docs : http://127.0.0.1:{server_port}/docs")


🚀 Server NLU Moderation berhasil berjalan di background!
📖 Swagger Docs : http://127.0.0.1:8001/docs


In [9]:
# CELL 9 — Demo Pengujian 12 Skenario Nyata (Pujian, Saran, Komplain, dan Sensor Kasar)

test_scenarios = [
    # ── Kategori: Kritik & Saran Positif
    ("Makanannya enak bgt, porsinya pas dan packing rapi!", "Kritik & Saran Positif"),
    ("Kopinya mantap, mungkin ke depannya bisa sediain varian oat milk ya kak", "Kritik & Saran Positif"),
    ("Saran saya packagingnya diganti paper bag ya biar lebih eco friendly", "Kritik & Saran Positif"),
    ("Pelayanan ramah banget, packagingnya rapi dan estetik.", "Kritik & Saran Positif"),
    ("Gila enak bgt parah nagih!", "Kritik & Saran Positif"),

    # ── Kategori: Kritik & Saran Negatif
    ("Ayamnya masih mentah bagian dalam kak, tolong lebih teliti lagi masaknya", "Kritik & Saran Negatif"),
    ("Pesanan sampai 1 jam lebih, makanannya sudah dingin dan kuahnya tumpah", "Kritik & Saran Negatif"),
    ("Adminnya judes banget dichat ga dibales", "Kritik & Saran Negatif"),
    ("porsinya dikit banget min, nyesel beli mahal-mahal", "Kritik & Saran Negatif"),

    # ── Kategori: Comment aneh-aneh (komentar yang berbahasa kasar)
    ("Woy babi lu jualan makanan basi anjing!", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("Penjual t0l0l, nyesel beli di toko sampah ini gblk", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("makanan baji***an lu nipu gue", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("b.a.b.i lu jualan kaga bener", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("4nj1nk lu gausah jualan", "Comment aneh-aneh (komentar yang berbahasa kasar)")
]

print("=" * 85)
print("🛡️ HASIL PENGUJIAN AKURASI MODEL NLU MODERATOR (TENANTIFY)")
print("=" * 85)

passed = 0
for text, expected in test_scenarios:
    res = classify_comment(text)
    actual = res["category"]
    action = res["action"]
    is_ok = (actual == expected)
    if is_ok:
        passed += 1
    
    icon = "✅" if is_ok else "❌"
    dash_status = "🟢 Masuk Dashboard" if res["is_delivered_to_dashboard"] else "🔴 Dihapus (Auto-Removed)"
    
    print(f"{icon} Input       : {text}")
    print(f"   Expected    : {expected}")
    print(f"   Hasil NLU   : {actual}")
    print(f"   Aksi Sistem : {action} ➔ {dash_status}")
    if res["aspects"]:
        print(f"   Aspek Topik : {', '.join(res['aspects'])}")
    if res["flagged_words"]:
        print(f"   Kata Kasar  : {', '.join(res['flagged_words'])}")
    print("-" * 85)

accuracy = (passed / len(test_scenarios)) * 100
print(f"\n🎯 TOTAL AKURASI: {passed}/{len(test_scenarios)} ({accuracy:.1f}%)")


🛡️ HASIL PENGUJIAN AKURASI MODEL NLU MODERATOR (TENANTIFY)
✅ Input       : Makanannya enak bgt, porsinya pas dan packing rapi!
   Expected    : Kritik & Saran Positif
   Hasil NLU   : Kritik & Saran Positif
   Aksi Sistem : DELIVERED_TO_DASHBOARD ➔ 🟢 Masuk Dashboard
   Aspek Topik : Rasa & Kualitas, Porsi & Takaran, Kemasan & Kebersihan
-------------------------------------------------------------------------------------
✅ Input       : Kopinya mantap, mungkin ke depannya bisa sediain varian oat milk ya kak
   Expected    : Kritik & Saran Positif
   Hasil NLU   : Kritik & Saran Positif
   Aksi Sistem : DELIVERED_TO_DASHBOARD ➔ 🟢 Masuk Dashboard
   Aspek Topik : Rasa & Kualitas
-------------------------------------------------------------------------------------
✅ Input       : Saran saya packagingnya diganti paper bag ya biar lebih eco friendly
   Expected    : Kritik & Saran Positif
   Hasil NLU   : Kritik & Saran Positif
   Aksi Sistem : DELIVERED_TO_DASHBOARD ➔ 🟢 Masuk Dashboard
   

In [10]:
# CELL 10 — Demo Batch Moderasi & Simulasi Tampilan Dashboard Penjual

sample_tenant_batch = [
    {"id": "rev-01", "tenantId": "tenant-1", "comment": "Kopi susunya creamy banget, next adain opsi less sugar ya!", "rating": 5.0},
    {"id": "rev-02", "tenantId": "tenant-1", "comment": "Woy kontol kopi apaan ini kemanisan anjink", "rating": 1.0},
    {"id": "rev-03", "tenantId": "tenant-1", "comment": "Roti bakarnya agak gosong pinggirannya kak, tolong diperbaiki", "rating": 2.5},
    {"id": "rev-04", "tenantId": "tenant-1", "comment": "Pelayanan mas baristanya ramah banget!", "rating": 5.0},
    {"id": "rev-05", "tenantId": "tenant-1", "comment": "b.a.b.i lu nipu pesanan kurang 1", "rating": 1.0}
]

batch_summary = moderate_batch(sample_tenant_batch)

print("📊 RINGKASAN MODERASI ULASAN TOKO:")
print(f" • Total Ulasan Masuk        : {batch_summary['total_processed']}")
print(f" • Diteruskan ke Dashboard   : {batch_summary['total_delivered_to_dashboard']} (Layak Tampil)")
print(f" • Otomatis Dihapus (Kasar)  : {batch_summary['total_auto_removed']} (Auto-Removed)")
print("\nDistribusi Kategori:")
for cat, count in batch_summary['summary_by_category'].items():
    print(f" - {cat}: {count}")

print("\n" + "=" * 60)
print("🏪 SIMULASI TAMPILAN DI DASHBOARD TOKO (Hanya Ulasan Lolos): ")
print("=" * 60)
for item in batch_summary['delivered_comments']:
    print(f"[{item['dashboard_badge']}] {item['original_text']}")
    print(f"   Topik: {item['aspects']} | Confidence: {item['confidence_score']}")
    print("-" * 60)


📊 RINGKASAN MODERASI ULASAN TOKO:
 • Total Ulasan Masuk        : 5
 • Diteruskan ke Dashboard   : 3 (Layak Tampil)
 • Otomatis Dihapus (Kasar)  : 2 (Auto-Removed)

Distribusi Kategori:
 - Kritik & Saran Positif: 2
 - Kritik & Saran Negatif: 1
 - Comment aneh-aneh (komentar yang berbahasa kasar): 2

🏪 SIMULASI TAMPILAN DI DASHBOARD TOKO (Hanya Ulasan Lolos): 
[Apresiasi & Rekomendasi] Kopi susunya creamy banget, next adain opsi less sugar ya!
   Topik: [] | Confidence: 0.97
------------------------------------------------------------
[Perlu Perhatian / Evaluasi] Roti bakarnya agak gosong pinggirannya kak, tolong diperbaiki
   Topik: ['Rasa & Kualitas'] | Confidence: 0.97
------------------------------------------------------------
[Apresiasi & Rekomendasi] Pelayanan mas baristanya ramah banget!
   Topik: ['Pelayanan & Tenant'] | Confidence: 0.97
------------------------------------------------------------


In [12]:
# CELL 11 — (Opsional) Public URL Tunnel via Ngrok
# Jalankan cell ini jika ingin menguji endpoint dari Postman / Frontend eksternal

from pyngrok import ngrok


ngrok.kill()

public_url = ngrok.connect(8001)
clean_url = public_url.public_url

print(f" Public API URL : {clean_url}")
print(f" Swagger Docs : {clean_url}/docs")


t=2026-11-09T12:39:49+0700 lvl=warn msg="can't bind default web address, trying alternatives" obj=web addr=127.0.0.1:4040
t=2026-11-09T12:39:49+0700 lvl=warn msg="failed to start tunnel" pg=/api/endpoints id=cc65db0c0a28c901 err="failed to start tunnel: The endpoint 'https://encircle-useable-unkind.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://encircle-useable-unkind.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}
